# Hito 2: Análisis Comparativo de Modelos

Este notebook reúne evidencia, análisis y material de apoyo para el **Hito 2** del proyecto de clasificación de imágenes. El objetivo no es reentrenar modelos, sino **analizar de forma reproducible** los artefactos ya generados para los dos enfoques comparados en el mismo problema multiclase single-label.

## Objetivo del Hito 2

- Resolver una tarea **multiclase single-label**.
- Considerar cada query del dataset como una clase distinta.
- Comparar dos enfoques con exactamente la misma partición `train/val/test`.
- Generar evidencia útil para el informe, la presentación y la defensa oral.

## Enfoques comparados

1. **CLIP Encoder + MLP**: usa embeddings preentrenados de `openai/clip-vit-base-patch32` y aprende una cabeza clasificadora multiclase.
2. **CNN**: aprende directamente desde imágenes redimensionadas a tamaño fijo (`128x128` por defecto en este proyecto).

## Criterio de justicia experimental

- Ambos enfoques usan el mismo dataset y el mismo esquema de etiquetas.
- Ambos reutilizan exactamente los mismos splits ya generados.
- Este notebook trabaja solo sobre **artefactos existentes** en `outputs/hito2/`.
- Si falta algún archivo, el notebook intenta seguir con el resto del análisis y deja una nota explícita.


In [14]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "outputs").exists():
            return candidate
    raise FileNotFoundError("No se pudo localizar la raíz del repositorio desde el directorio actual.")


REPO_ROOT = find_repo_root()
CACHE_ROOT = REPO_ROOT / 'outputs' / '.hito2_cache'
(CACHE_ROOT / 'matplotlib').mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str((CACHE_ROOT / 'matplotlib').resolve()))
os.environ.setdefault('XDG_CACHE_HOME', str(CACHE_ROOT.resolve()))

import matplotlib
matplotlib.use('Agg')
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass

    def display(value):
        print(value)


os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 120)

from src.hito2.config import (
    HITO2_CLIP_MLP_OUTPUT_DIR,
    HITO2_CNN_OUTPUT_DIR,
    HITO2_COMPARISON_OUTPUT_DIR,
    HITO2_MANIFEST_DIR,
    HITO2_SPLIT_DIR,
)
from src.hito2.notebook_analysis import (
    build_class_distribution_dataframe,
    build_confusion_matrix_from_predictions,
    build_dataset_overview,
    build_metrics_comparison_dataframe,
    build_per_class_comparison_dataframe,
    build_top_confusions_dataframe,
    build_top_k_gain_dataframe,
    load_experiment_bundle,
    load_shared_artifacts,
    select_prediction_examples,
)

print(f'Raíz del repositorio: {REPO_ROOT}')


Raíz del repositorio: /Users/enzopalavicino/Desktop/Artificial Intelligence/Repo/IA_Bertens_Palavicino


## 2. Carga de artefactos

Esta sección carga automáticamente los artefactos más importantes de ambos experimentos:

- métricas de validación y test,
- historial de entrenamiento por época,
- reports por clase,
- tablas de predicciones,
- matrices de confusión ya exportadas,
- manifest y splits del Hito 2,
- y la comparación consolidada, si ya existe.


In [15]:
paths = {
    "clip_dir": HITO2_CLIP_MLP_OUTPUT_DIR,
    "cnn_dir": HITO2_CNN_OUTPUT_DIR,
    "manifest_path": HITO2_MANIFEST_DIR / "query_manifest.csv",
    "split_path": HITO2_SPLIT_DIR / "query_splits.csv",
    "split_summary_path": HITO2_SPLIT_DIR / "query_splits_summary.json",
    "comparison_path": HITO2_COMPARISON_OUTPUT_DIR / "comparison_metrics.csv",
}

clip_bundle = load_experiment_bundle("clip_mlp", paths["clip_dir"], display_name="CLIP + MLP")
cnn_bundle = load_experiment_bundle("cnn", paths["cnn_dir"], display_name="CNN")
experiments = [clip_bundle, cnn_bundle]

shared = load_shared_artifacts(
    manifest_path=paths["manifest_path"],
    split_path=paths["split_path"],
    split_summary_path=paths["split_summary_path"],
    comparison_path=paths["comparison_path"],
)
manifest_df = shared["manifest_df"]
split_df = shared["split_df"]
split_summary = shared["split_summary"]
comparison_df_disk = shared["comparison_df"]

artifact_status_rows = []
for bundle in experiments:
    artifact_status_rows.append(
        {
            "model": bundle.display_name,
            "experiment_dir": bundle.experiment_dir.relative_to(REPO_ROOT).as_posix(),
            "missing_artifacts": len(bundle.missing_artifacts),
            "best_epoch": bundle.best_epoch,
            "val_predictions": len(bundle.val_predictions),
            "test_predictions": len(bundle.test_predictions),
        }
    )

artifact_status_df = pd.DataFrame(artifact_status_rows)
display(artifact_status_df)

missing_notes = []
for bundle in experiments:
    if bundle.missing_artifacts:
        missing_notes.append(f"- **{bundle.display_name}**: " + ", ".join(bundle.missing_artifacts))

if missing_notes:
    note_text = "**Nota de robustez.** Se detectaron artefactos faltantes, pero el notebook seguirá con la información disponible.\n\n" + "\n".join(missing_notes)
    display(Markdown(note_text))
else:
    display(Markdown("**Carga exitosa.** Se encontraron los artefactos principales de ambos experimentos."))

print(f"Manifest: {len(manifest_df)} filas")
print(f"Splits: {len(split_df)} filas")
print(f"Comparación consolidada cargada desde disco: {not comparison_df_disk.empty}")


,model,experiment_dir,missing_artifacts,best_epoch,val_predictions,test_predictions
0,CLIP + MLP,outputs/hito2/clip_mlp,0,40,221,221
1,CNN,outputs/hito2/cnn,0,22,221,221


**Carga exitosa.** Se encontraron los artefactos principales de ambos experimentos.

Manifest: 1447 filas
Splits: 1447 filas
Comparación consolidada cargada desde disco: True


## 3. Resumen general del dataset y setup experimental

Antes de comparar modelos, conviene recordar el contexto del problema:

- el dataset es **multiclase single-label**,
- cada query corresponde a una clase,
- la comparación se hace sobre **35 clases**,
- y el dataset es marcadamente **desbalanceado**.

Esto importa especialmente para interpretar métricas como **macro F1**, donde las clases raras pesan igual que las frecuentes.


In [16]:
dataset_overview = build_dataset_overview(manifest_df, split_df)
class_distribution_df = build_class_distribution_dataframe(manifest_df, split_df)

split_counts_df = pd.DataFrame(columns=["split", "num_images"])
if not split_df.empty and "split" in split_df.columns:
    split_counts_df = (
        split_df["split"].value_counts().rename_axis("split").reset_index(name="num_images").sort_values("split")
    )

rare_classes_df = pd.DataFrame(columns=["class_name", "total_images", "train", "val", "test"])
if not class_distribution_df.empty:
    rare_classes_df = class_distribution_df[class_distribution_df["total_images"] <= 5].copy()
    rare_classes_df = rare_classes_df.sort_values(["total_images", "class_name"], ascending=[True, True])

fallback_classes = split_summary.get("fallback_classes", []) if isinstance(split_summary, dict) else []
classes_missing_from_val = split_summary.get("classes_missing_from_val", []) if isinstance(split_summary, dict) else []

overview_md = f'''
### Resumen cuantitativo

- **Número total de imágenes válidas:** {dataset_overview.get("num_images", 0)}
- **Número de clases:** {dataset_overview.get("num_classes", 0)}
- **Imágenes en train:** {dataset_overview.get("split_counts", {}).get("train", 0)}
- **Imágenes en val:** {dataset_overview.get("split_counts", {}).get("val", 0)}
- **Imágenes en test:** {dataset_overview.get("split_counts", {}).get("test", 0)}
- **Clases con fallback por escasez:** {", ".join(fallback_classes) if fallback_classes else "ninguna"}
- **Clases ausentes en validación:** {", ".join(classes_missing_from_val) if classes_missing_from_val else "ninguna"}
'''
display(Markdown(overview_md))

display(split_counts_df)
display(class_distribution_df[["class_name", "class_id", "total_images", "train", "val", "test"]].head(12))

if not rare_classes_df.empty:
    display(Markdown("### Clases muy escasas (5 imágenes o menos)"))
    display(rare_classes_df[["class_name", "total_images", "train", "val", "test"]])
else:
    display(Markdown("No se detectaron clases con 5 imágenes o menos."))



### Resumen cuantitativo

- **Número total de imágenes válidas:** 1447
- **Número de clases:** 35
- **Imágenes en train:** 1005
- **Imágenes en val:** 221
- **Imágenes en test:** 221
- **Clases con fallback por escasez:** obj_3, obj_36, obj_37, obj_42
- **Clases ausentes en validación:** obj_3, obj_36, obj_37, obj_42


,split,num_images
2,test,221
0,train,1005
1,val,221


,class_name,class_id,total_images,train,val,test
0,marqeur,14,409,286,62,61
1,simple_sep,32,160,112,24,24
2,S,2,147,103,22,22
3,losange,13,117,82,18,17
4,croix,7,92,64,14,14
5,double_sep,8,87,61,13,13
6,rubanlettrine,30,68,48,10,10
7,encadrement,9,60,42,9,9
8,T,3,39,27,6,6
9,D,1,35,25,5,5


### Clases muy escasas (5 imágenes o menos)

,class_name,total_images,train,val,test
31,obj_3,2,1,0,1
32,obj_36,2,1,0,1
33,obj_37,2,1,0,1
34,obj_42,2,1,0,1
28,henri_g,3,1,1,1
29,obj_39,3,1,1,1
30,rubanlettrine_b,3,1,1,1
24,obj_1,4,2,1,1
25,obj_31,4,2,1,1
26,obj_35,4,2,1,1


In [17]:
if class_distribution_df.empty:
    print("No hay datos de distribución por clase para graficar.")
else:
    plot_df = class_distribution_df.sort_values("total_images", ascending=True).copy()
    rare_mask = plot_df["total_images"] <= 5
    colors = np.where(rare_mask, "#d55e00", "#4c78a8")

    fig, axes = plt.subplots(1, 2, figsize=(20, 14), gridspec_kw={"width_ratios": [1.0, 1.25]})

    axes[0].barh(plot_df["class_name"], plot_df["total_images"], color=colors)
    axes[0].set_title("Distribución total por clase")
    axes[0].set_xlabel("Número de imágenes")
    axes[0].axvline(5, color="black", linestyle="--", linewidth=1, alpha=0.7)
    axes[0].text(5.2, 0.5, "umbral de rareza = 5", fontsize=10, alpha=0.8)

    axes[1].barh(plot_df["class_name"], plot_df["train"], label="train", color="#4c78a8")
    axes[1].barh(plot_df["class_name"], plot_df["val"], left=plot_df["train"], label="val", color="#f58518")
    axes[1].barh(
        plot_df["class_name"],
        plot_df["test"],
        left=plot_df["train"] + plot_df["val"],
        label="test",
        color="#54a24b",
    )
    axes[1].set_title("Composición por split")
    axes[1].set_xlabel("Número de imágenes")
    axes[1].legend(loc="lower right")

    plt.tight_layout()
    plt.show()


/var/folders/_v/tt01wjhj06jdp6jd9t6rqh9r0000gn/T/ipykernel_46638/2994303979.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Lectura del desbalance.** La distribución por clase no es uniforme: existen clases muy frecuentes y varias clases con muy pocas imágenes. Esto vuelve especialmente relevante el uso de métricas como **macro precision**, **macro recall** y **macro F1**, porque penalizan más claramente cuando un modelo falla en clases minoritarias. También explica por qué algunas clases aparecen con soporte cero en validación: con solo dos imágenes totales, no es posible repartirlas en `train/val/test` sin romper la consistencia del split.


## 4. Tabla comparativa principal de métricas

A continuación se comparan ambos enfoques con las métricas más importantes del Hito 2. La tabla usa los artefactos ya guardados por cada experimento y, cuando existe, se puede contrastar con la comparación consolidada exportada en `outputs/hito2/comparison/`.


In [18]:
metrics_df = build_metrics_comparison_dataframe(experiments)
metric_order = [
    "best_epoch",
    "epochs_ran",
    "val_accuracy",
    "test_accuracy",
    "val_macro_precision",
    "test_macro_precision",
    "val_macro_recall",
    "test_macro_recall",
    "val_macro_f1",
    "test_macro_f1",
    "val_weighted_f1",
    "test_weighted_f1",
    "val_top_3_accuracy",
    "test_top_3_accuracy",
    "val_loss",
    "test_loss",
]
metrics_view = metrics_df[[column for column in metric_order if column in metrics_df.columns]].copy()

float_columns = [column for column in metrics_view.columns if column not in {"best_epoch", "epochs_ran"}]
highlight_max_columns = [
    column for column in metrics_view.columns
    if column.endswith(("accuracy", "precision", "recall", "f1"))
]
highlight_min_columns = [column for column in metrics_view.columns if column.endswith("loss")]

styled_metrics = metrics_view.style.format({column: "{:.4f}" for column in float_columns}, na_rep="—")
if "best_epoch" in metrics_view.columns:
    styled_metrics = styled_metrics.format({"best_epoch": "{:.0f}"}, na_rep="—")
if "epochs_ran" in metrics_view.columns:
    styled_metrics = styled_metrics.format({"epochs_ran": "{:.0f}"}, na_rep="—")
if highlight_max_columns:
    styled_metrics = styled_metrics.highlight_max(subset=highlight_max_columns, color="#d8f3dc")
if highlight_min_columns:
    styled_metrics = styled_metrics.highlight_min(subset=highlight_min_columns, color="#fde2e4")
display(styled_metrics)

if not comparison_df_disk.empty:
    display(Markdown("La misma comparación global ya existe en `outputs/hito2/comparison/comparison_metrics.csv`; aquí se vuelve a mostrar con foco analítico para el notebook."))

test_accuracy_winner = metrics_df["test_accuracy"].idxmax()
test_macro_f1_winner = metrics_df["test_macro_f1"].idxmax()
acc_gap = metrics_df.loc[test_accuracy_winner, "test_accuracy"] - metrics_df.drop(index=test_accuracy_winner)["test_accuracy"].max()
macro_f1_gap = metrics_df.loc[test_macro_f1_winner, "test_macro_f1"] - metrics_df.drop(index=test_macro_f1_winner)["test_macro_f1"].max()

display(
    Markdown(
        f'''**Interpretación breve.** En los artefactos disponibles, **{test_accuracy_winner}** obtiene la mejor `test accuracy` y **{test_macro_f1_winner}** logra el mejor `test macro F1`. La diferencia en accuracy respecto al otro enfoque es de **{acc_gap:.4f}**, mientras que la diferencia en macro F1 es de **{macro_f1_gap:.4f}**. La lectura conjunta de `macro F1` y `weighted F1` ayuda a distinguir entre rendimiento global y comportamiento sobre clases minoritarias.'''
    )
)


,best_epoch,epochs_ran,val_accuracy,test_accuracy,val_macro_precision,test_macro_precision,val_macro_recall,test_macro_recall,val_macro_f1,test_macro_f1,val_weighted_f1,test_weighted_f1,val_top_3_accuracy,test_top_3_accuracy,val_loss,test_loss
model,,,,,,,,,,,,,,,,
CLIP + MLP,40,48,0.846154,0.873303,0.679942,0.727767,0.723981,0.766183,0.683359,0.733593,0.833651,0.858411,0.986425,0.990950,0.685367,0.825759
CNN,22,28,0.819005,0.755656,0.574702,0.555159,0.669626,0.630305,0.604626,0.569102,0.816969,0.760208,0.977376,0.963801,0.722518,0.899608


La misma comparación global ya existe en `outputs/hito2/comparison/comparison_metrics.csv`; aquí se vuelve a mostrar con foco analítico para el notebook.

**Interpretación breve.** En los artefactos disponibles, **CLIP + MLP** obtiene la mejor `test accuracy` y **CLIP + MLP** logra el mejor `test macro F1`. La diferencia en accuracy respecto al otro enfoque es de **0.1176**, mientras que la diferencia en macro F1 es de **0.1645**. La lectura conjunta de `macro F1` y `weighted F1` ayuda a distinguir entre rendimiento global y comportamiento sobre clases minoritarias.

## 5. Curvas de entrenamiento

Inspirados en los laboratorios de MLP y anatomía del entrenamiento, conviene revisar cómo evolucionan pérdida y accuracy en train/val. Esto permite discutir convergencia, estabilidad y señales de sobreajuste sin necesidad de reentrenar nada.


In [19]:
def plot_training_curves(bundle):
    history_df = bundle.history_df.copy()
    if history_df.empty:
        print(f"No hay historial de entrenamiento para {bundle.display_name}.")
        return

    fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True)
    epoch_values = history_df["epoch"]

    axes[0, 0].plot(epoch_values, history_df["train_loss"], label="train", linewidth=2)
    axes[0, 0].plot(epoch_values, history_df["val_loss"], label="val", linewidth=2)
    axes[0, 0].set_title(f"{bundle.display_name}: loss")
    axes[0, 0].set_ylabel("Loss")
    axes[0, 0].legend()

    axes[0, 1].plot(epoch_values, history_df["train_accuracy"], label="train", linewidth=2)
    axes[0, 1].plot(epoch_values, history_df["val_accuracy"], label="val", linewidth=2)
    axes[0, 1].set_title(f"{bundle.display_name}: accuracy")
    axes[0, 1].set_ylabel("Accuracy")
    axes[0, 1].legend()

    axes[1, 0].plot(epoch_values, history_df["val_macro_f1"], color="#54a24b", linewidth=2)
    axes[1, 0].set_title(f"{bundle.display_name}: val macro F1")
    axes[1, 0].set_ylabel("Macro F1")
    axes[1, 0].set_xlabel("Época")

    axes[1, 1].plot(epoch_values, history_df["val_weighted_f1"], color="#e45756", linewidth=2)
    axes[1, 1].set_title(f"{bundle.display_name}: val weighted F1")
    axes[1, 1].set_ylabel("Weighted F1")
    axes[1, 1].set_xlabel("Época")

    for ax in axes.flat:
        ax.grid(alpha=0.25)

    plt.tight_layout()
    plt.show()


for bundle in experiments:
    plot_training_curves(bundle)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharex=False)
for bundle in experiments:
    history_df = bundle.history_df.copy()
    if history_df.empty:
        continue
    axes[0].plot(history_df["epoch"], history_df["val_loss"], linewidth=2, label=bundle.display_name)
    axes[1].plot(history_df["epoch"], history_df["val_accuracy"], linewidth=2, label=bundle.display_name)

axes[0].set_title("Comparación directa: validation loss")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Val loss")
axes[0].legend()

axes[1].set_title("Comparación directa: validation accuracy")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Val accuracy")
axes[1].legend()

for ax in axes:
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()


/var/folders/_v/tt01wjhj06jdp6jd9t6rqh9r0000gn/T/ipykernel_46638/2328725465.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/_v/tt01wjhj06jdp6jd9t6rqh9r0000gn/T/ipykernel_46638/2328725465.py:64: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
stability_rows = []
for bundle in experiments:
    history_df = bundle.history_df.copy()
    if history_df.empty:
        continue
    val_loss_std = float(history_df["val_loss"].astype(float).std(ddof=0))
    mean_abs_delta = float(history_df["val_loss"].astype(float).diff().abs().dropna().mean()) if len(history_df) > 1 else 0.0
    final_gap = float(history_df["train_accuracy"].iloc[-1] - history_df["val_accuracy"].iloc[-1])
    stability_rows.append(
        {
            "model": bundle.display_name,
            "best_epoch": bundle.best_epoch,
            "epochs_ran": len(history_df),
            "best_val_loss": float(history_df["val_loss"].min()),
            "final_val_loss": float(history_df["val_loss"].iloc[-1]),
            "val_loss_std": val_loss_std,
            "mean_abs_val_loss_delta": mean_abs_delta,
            "final_train_minus_val_accuracy": final_gap,
        }
    )

stability_df = pd.DataFrame(stability_rows).set_index("model")
display(stability_df.style.format("{:.4f}", na_rep="—"))

if not stability_df.empty:
    most_stable = stability_df["val_loss_std"].idxmin()
    fastest = stability_df["best_epoch"].idxmin()
    display(
        Markdown(
            f'''**Lectura de estabilidad.** Bajo una heurística simple, **{most_stable}** muestra menor variabilidad en `val_loss`, mientras que **{fastest}** alcanza su mejor checkpoint antes. Conviene combinar esta lectura con las métricas finales: converger antes no implica necesariamente generalizar mejor.'''
        )
    )


,best_epoch,epochs_ran,best_val_loss,final_val_loss,val_loss_std,mean_abs_val_loss_delta,final_train_minus_val_accuracy
model,,,,,,,
CLIP + MLP,40.0000,48.0000,0.6854,0.7396,0.7450,0.1099,0.0684
CNN,22.0000,28.0000,0.7225,0.7324,0.6038,0.5485,0.1284


**Lectura de estabilidad.** Bajo una heurística simple, **CNN** muestra menor variabilidad en `val_loss`, mientras que **CNN** alcanza su mejor checkpoint antes. Conviene combinar esta lectura con las métricas finales: converger antes no implica necesariamente generalizar mejor.

**Qué mirar en las curvas.** Si `train loss` sigue bajando mientras `val loss` se estanca o empeora, aparecen señales de sobreajuste. Si ambos descienden de manera razonablemente coordinada, la convergencia es más sana. En este problema, además, la lectura de `val macro F1` es clave porque refleja mejor qué pasa con clases minoritarias que la accuracy sola.


## 6. Comparación de matrices de confusión

Las matrices de confusión permiten ver si los errores están más dispersos o más concentrados en ciertas parejas de clases. Si el PNG ya fue exportado por el experimento, se usa directamente; si no, el notebook intenta reconstruir la matriz a partir de las predicciones guardadas.


In [21]:
def draw_confusion_panel(ax, bundle, split_name: str):
    image_path = bundle.val_confusion_path if split_name == "val" else bundle.test_confusion_path
    if image_path is not None and image_path.exists():
        ax.imshow(mpimg.imread(image_path))
        ax.axis("off")
        ax.set_title(f"{bundle.display_name} - {split_name}")
        return

    predictions_df = bundle.val_predictions if split_name == "val" else bundle.test_predictions
    class_names = bundle.class_names
    if predictions_df.empty or not class_names:
        ax.axis("off")
        ax.text(0.5, 0.5, "Sin datos suficientes", ha="center", va="center")
        ax.set_title(f"{bundle.display_name} - {split_name}")
        return

    confusion = build_confusion_matrix_from_predictions(predictions_df, class_names)
    sns.heatmap(confusion, ax=ax, cmap="Blues", cbar=False, xticklabels=False, yticklabels=False)
    ax.set_title(f"{bundle.display_name} - {split_name}")
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Clase real")


fig, axes = plt.subplots(2, 2, figsize=(18, 16))
draw_confusion_panel(axes[0, 0], clip_bundle, "val")
draw_confusion_panel(axes[0, 1], clip_bundle, "test")
draw_confusion_panel(axes[1, 0], cnn_bundle, "val")
draw_confusion_panel(axes[1, 1], cnn_bundle, "test")
plt.tight_layout()
plt.show()


/var/folders/_v/tt01wjhj06jdp6jd9t6rqh9r0000gn/T/ipykernel_46638/44206996.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [22]:
def confusion_dispersion_summary(predictions_df: pd.DataFrame) -> dict:
    if predictions_df.empty:
        return {"num_errors": 0, "unique_pairs": 0, "top_5_mass": 0.0}
    errors_df = predictions_df[predictions_df["true_class_name"] != predictions_df["pred_class_name"]].copy()
    if errors_df.empty:
        return {"num_errors": 0, "unique_pairs": 0, "top_5_mass": 0.0}
    pair_counts = errors_df.groupby(["true_class_name", "pred_class_name"]).size().sort_values(ascending=False)
    return {
        "num_errors": int(len(errors_df)),
        "unique_pairs": int(len(pair_counts)),
        "top_5_mass": float(pair_counts.head(5).sum() / pair_counts.sum()),
    }


confusion_tables = []
for bundle in experiments:
    top_confusions_df = build_top_confusions_dataframe(bundle.test_predictions, top_n=10)
    top_confusions_df.insert(0, "model", bundle.display_name)
    confusion_tables.append(top_confusions_df)

top_confusions_test_df = pd.concat(confusion_tables, ignore_index=True) if confusion_tables else pd.DataFrame()
display(top_confusions_test_df)

clip_dispersion = confusion_dispersion_summary(clip_bundle.test_predictions)
cnn_dispersion = confusion_dispersion_summary(cnn_bundle.test_predictions)
more_concentrated = "CLIP + MLP" if clip_dispersion["top_5_mass"] > cnn_dispersion["top_5_mass"] else "CNN"

display(
    Markdown(
        f'''
        ### Lectura comparativa de errores

        - **CLIP + MLP**: {clip_dispersion["num_errors"]} errores en test, {clip_dispersion["unique_pairs"]} parejas de confusión distintas, masa de top-5 = {clip_dispersion["top_5_mass"]:.3f}.
        - **CNN**: {cnn_dispersion["num_errors"]} errores en test, {cnn_dispersion["unique_pairs"]} parejas de confusión distintas, masa de top-5 = {cnn_dispersion["top_5_mass"]:.3f}.
        - Con esta heurística, los errores están **más concentrados** en **{more_concentrated}** cuando su masa en las cinco parejas más frecuentes es mayor. Menos errores totales y una dispersión controlada suelen ser una señal favorable para la generalización práctica.
        '''
    )
)


,model,true_class_name,pred_class_name,count
0,CLIP + MLP,simple_sep,double_sep,10
1,CLIP + MLP,triple_sep,double_sep,3
2,CLIP + MLP,T,S,2
3,CLIP + MLP,double_sep,simple_sep,2
4,CLIP + MLP,T,croix,1
5,CLIP + MLP,bateau,bateau_g,1
6,CLIP + MLP,bateau_d,bateau_g,1
7,CLIP + MLP,bateau_g,encadrement,1
8,CLIP + MLP,croix,T,1
9,CLIP + MLP,henri_g,henri_d,1



        ### Lectura comparativa de errores

        - **CLIP + MLP**: 28 errores en test, 15 parejas de confusión distintas, masa de top-5 = 0.643.
        - **CNN**: 54 errores en test, 29 parejas de confusión distintas, masa de top-5 = 0.463.
        - Con esta heurística, los errores están **más concentrados** en **CLIP + MLP** cuando su masa en las cinco parejas más frecuentes es mayor. Menos errores totales y una dispersión controlada suelen ser una señal favorable para la generalización práctica.
        

## 7. Análisis por clase

El análisis agregado no basta para entender qué ocurre con clases específicas. Por eso aquí se comparan `precision`, `recall` y `F1` por clase usando los `classification_report` ya exportados por cada pipeline.


In [23]:
per_class_test_df = build_per_class_comparison_dataframe(experiments, split_name="test")
per_class_val_df = build_per_class_comparison_dataframe(experiments, split_name="val")

display(Markdown("### Comparación por clase en test"))
display(per_class_test_df.head(15))

if not per_class_test_df.empty and {"clip_mlp_f1", "cnn_f1"}.issubset(per_class_test_df.columns):
    top_clip_adv_df = per_class_test_df.sort_values("f1_gap", ascending=False).head(10)
    top_cnn_adv_df = per_class_test_df.sort_values("f1_gap", ascending=True).head(10)

    display(Markdown("### Clases donde CLIP + MLP supera más claramente a CNN"))
    display(top_clip_adv_df[["class_name", "support", "clip_mlp_f1", "cnn_f1", "f1_gap"]])

    display(Markdown("### Clases donde CNN se acerca más o supera a CLIP + MLP"))
    display(top_cnn_adv_df[["class_name", "support", "clip_mlp_f1", "cnn_f1", "f1_gap"]])

    strongest_classes_df = per_class_test_df.assign(best_f1=per_class_test_df[["clip_mlp_f1", "cnn_f1"]].max(axis=1))
    strongest_classes_df = strongest_classes_df.sort_values(["best_f1", "support"], ascending=[False, False]).head(10)

    weakest_classes_df = per_class_test_df.assign(mean_f1=per_class_test_df[["clip_mlp_f1", "cnn_f1"]].mean(axis=1))
    weakest_classes_df = weakest_classes_df.sort_values(["mean_f1", "support"], ascending=[True, True]).head(10)

    display(Markdown("### Top 10 clases más fuertes"))
    display(strongest_classes_df[["class_name", "support", "clip_mlp_f1", "cnn_f1", "best_f1"]])

    display(Markdown("### Top 10 clases más débiles"))
    display(weakest_classes_df[["class_name", "support", "clip_mlp_f1", "cnn_f1", "mean_f1"]])

    plot_df = per_class_test_df.sort_values(["support", "class_name"], ascending=[False, True]).copy()
    positions = np.arange(len(plot_df))

    fig, ax = plt.subplots(figsize=(20, 7))
    ax.bar(positions - 0.2, plot_df["clip_mlp_f1"], width=0.4, label="CLIP + MLP", color="#4c78a8")
    ax.bar(positions + 0.2, plot_df["cnn_f1"], width=0.4, label="CNN", color="#f58518")
    ax.set_xticks(positions)
    ax.set_xticklabels(plot_df["class_name"], rotation=90)
    ax.set_ylabel("F1 en test")
    ax.set_title("Comparación de F1 por clase")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    display(Markdown("No hay suficiente información por clase para comparar F1."))


### Comparación por clase en test

,class_name,clip_mlp_precision,clip_mlp_recall,clip_mlp_f1,clip_mlp_support,cnn_precision,cnn_recall,cnn_f1,cnn_support,support,f1_gap
14,marqeur,1.000000,1.000000,1.000000,61,1.000000,0.901639,0.948276,61,61,0.051724
32,simple_sep,0.812500,0.541667,0.650000,24,0.941176,0.666667,0.780488,24,24,-0.130488
2,S,0.880000,1.000000,0.936170,22,0.772727,0.772727,0.772727,22,22,0.163443
13,losange,1.000000,1.000000,1.000000,17,1.000000,0.764706,0.866667,17,17,0.133333
7,croix,0.928571,0.928571,0.928571,14,0.500000,0.285714,0.363636,14,14,0.564935
8,double_sep,0.458333,0.846154,0.594595,13,0.500000,0.769231,0.606061,13,13,-0.011466
30,rubanlettrine,0.909091,1.000000,0.952381,10,0.900000,0.900000,0.900000,10,10,0.052381
9,encadrement,0.900000,1.000000,0.947368,9,0.900000,1.000000,0.947368,9,9,0.000000
3,T,0.750000,0.500000,0.600000,6,0.250000,0.500000,0.333333,6,6,0.266667
1,D,1.000000,1.000000,1.000000,5,1.000000,1.000000,1.000000,5,5,0.000000


### Clases donde CLIP + MLP supera más claramente a CNN

,class_name,support,clip_mlp_f1,cnn_f1,f1_gap
25,obj_40,1,1.000000,0.000000,1.000000
21,obj_36,1,1.000000,0.000000,1.000000
27,obj_61,1,0.666667,0.000000,0.666667
23,obj_38,1,0.666667,0.000000,0.666667
11,henri_d,1,0.666667,0.000000,0.666667
10,grand_A,2,1.000000,0.400000,0.600000
7,croix,14,0.928571,0.363636,0.564935
6,bateau_g,2,0.400000,0.000000,0.400000
0,BP,2,1.000000,0.666667,0.333333
18,obj_31,1,1.000000,0.666667,0.333333


### Clases donde CNN se acerca más o supera a CLIP + MLP

...

### Top 10 clases más fuertes

,class_name,support,clip_mlp_f1,cnn_f1,best_f1
14,marqeur,61,1.0,0.948276,1.0
13,losange,17,1.0,0.866667,1.0
1,D,5,1.0,1.000000,1.0
29,petit_A,5,1.0,1.000000,1.0
28,pdp,4,1.0,0.800000,1.0
10,grand_A,2,1.0,0.400000,1.0
0,BP,2,1.0,0.666667,1.0
33,status,2,1.0,0.800000,1.0
21,obj_36,1,1.0,0.000000,1.0
25,obj_40,1,1.0,0.000000,1.0


### Top 10 clases más débiles

,class_name,support,clip_mlp_f1,cnn_f1,mean_f1
5,bateau_d,1,0.000000,0.000000,0.000000
15,obj_1,1,0.000000,0.000000,0.000000
31,rubanlettrine_b,1,0.000000,0.000000,0.000000
6,bateau_g,2,0.400000,0.000000,0.200000
34,triple_sep,4,0.000000,0.400000,0.200000
11,henri_d,1,0.666667,0.000000,0.333333
23,obj_38,1,0.666667,0.000000,0.333333
27,obj_61,1,0.666667,0.000000,0.333333
12,henri_g,1,0.000000,0.666667,0.333333
26,obj_42,1,0.000000,0.666667,0.333333


/var/folders/_v/tt01wjhj06jdp6jd9t6rqh9r0000gn/T/ipykernel_46638/1820950942.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Cómo leer esta sección.** Las clases fuertes suelen tener más soporte y patrones visuales más consistentes. Las clases débiles pueden fallar por soporte escaso, alta similitud visual con otras clases o poca variedad en entrenamiento. Una ventaja grande en `macro F1` suele reflejar justamente que un modelo gestiona mejor este tipo de clases frágiles.


## 8. Top-1 vs Top-3

Además de la accuracy tradicional, ambos experimentos reportan **top-3 accuracy**. Esta métrica es útil cuando varias clases son visualmente parecidas o cuando interesa que la clase correcta aparezca dentro de un pequeño conjunto de hipótesis plausibles.


In [24]:
topk_df = build_top_k_gain_dataframe(experiments)
display(topk_df.style.format("{:.4f}", na_rep="—"))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
positions = np.arange(len(topk_df.index))
width = 0.35

axes[0].bar(positions - width / 2, topk_df["val_top_1"], width=width, label="Top-1", color="#4c78a8")
axes[0].bar(positions + width / 2, topk_df["val_top_3"], width=width, label="Top-3", color="#54a24b")
axes[0].set_xticks(positions)
axes[0].set_xticklabels(topk_df.index)
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Validación: Top-1 vs Top-3")
axes[0].legend()

axes[1].bar(positions - width / 2, topk_df["test_top_1"], width=width, label="Top-1", color="#f58518")
axes[1].bar(positions + width / 2, topk_df["test_top_3"], width=width, label="Top-3", color="#e45756")
axes[1].set_xticks(positions)
axes[1].set_xticklabels(topk_df.index)
axes[1].set_ylim(0, 1.05)
axes[1].set_title("Test: Top-1 vs Top-3")
axes[1].legend()

plt.tight_layout()
plt.show()

best_top3_model = topk_df["test_top_3"].idxmax()
best_top1_model = topk_df["test_top_1"].idxmax()
display(
    Markdown(
        f'''**Interpretación.** En test, **{best_top1_model}** lidera en top-1 y **{best_top3_model}** también lidera en top-3. La brecha entre top-1 y top-3 muestra cuánto mejora cada enfoque cuando se permite considerar varias clases candidatas, algo útil en un problema con clases visualmente cercanas y distribución desbalanceada.'''
    )
)


,val_top_1,val_top_3,val_gain,test_top_1,test_top_3,test_gain
model,,,,,,
CLIP + MLP,0.8462,0.9864,0.1403,0.8733,0.9910,0.1176
CNN,0.8190,0.9774,0.1584,0.7557,0.9638,0.2081


/var/folders/_v/tt01wjhj06jdp6jd9t6rqh9r0000gn/T/ipykernel_46638/1507349684.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretación.** En test, **CLIP + MLP** lidera en top-1 y **CLIP + MLP** también lidera en top-3. La brecha entre top-1 y top-3 muestra cuánto mejora cada enfoque cuando se permite considerar varias clases candidatas, algo útil en un problema con clases visualmente cercanas y distribución desbalanceada.

## 9. Ejemplos de predicciones correctas e incorrectas

A continuación se muestran ejemplos seleccionados desde las predicciones guardadas. La idea no es saturar el notebook, sino ofrecer evidencia visual de aciertos y errores en el conjunto de test.


In [25]:
def show_prediction_examples(bundle, split_name: str = "test", correct: bool = True, n: int = 3):
    predictions_df = bundle.test_predictions if split_name == "test" else bundle.val_predictions
    selected_df = select_prediction_examples(predictions_df, correct=correct, n=n)
    section_name = "aciertos" if correct else "errores"

    if selected_df.empty:
        display(Markdown(f"No hay {section_name} disponibles para {bundle.display_name} en {split_name}."))
        return

    fig, axes = plt.subplots(1, len(selected_df), figsize=(5 * len(selected_df), 4))
    if len(selected_df) == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, selected_df.iterrows()):
        image_path = REPO_ROOT / str(row["image_path"])
        if image_path.exists():
            ax.imshow(mpimg.imread(image_path))
        else:
            ax.text(0.5, 0.5, "Imagen no disponible", ha="center", va="center")
        ax.axis("off")

        top3_parts = []
        for rank in (1, 2, 3):
            class_col = f"top_{rank}_class_name"
            prob_col = f"top_{rank}_probability"
            if class_col in row.index and prob_col in row.index:
                top3_parts.append(f"{row[class_col]} ({row[prob_col]:.2f})")
        top3_text = " | ".join(top3_parts)

        ax.set_title(
            f"Real: {row['true_class_name']}\nPred: {row['pred_class_name']}\nTop-3: {top3_text}",
            fontsize=10,
        )

    plt.suptitle(f"{bundle.display_name} - {split_name} - {section_name}", y=1.02)
    plt.tight_layout()
    plt.show()


for bundle in experiments:
    show_prediction_examples(bundle, split_name="test", correct=True, n=3)
    show_prediction_examples(bundle, split_name="test", correct=False, n=3)


/var/folders/_v/tt01wjhj06jdp6jd9t6rqh9r0000gn/T/ipykernel_46638/3779756048.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/_v/tt01wjhj06jdp6jd9t6rqh9r0000gn/T/ipykernel_46638/3779756048.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/_v/tt01wjhj06jdp6jd9t6rqh9r0000gn/T/ipykernel_46638/3779756048.py:10: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(1, len(selected_df), figsize=(5 * len(selected_df), 4))
/var/folders/_v/tt01wjhj06jdp6jd9t6rqh9r0000gn/T/ipykernel_46638/3779756048.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folder

## 10. Discusión metodológica

Desde el punto de vista conceptual, los dos enfoques comparados aprenden de maneras distintas.

- **CLIP + MLP** separa el problema en dos etapas: primero usa un encoder visual preentrenado para obtener una representación semántica rica de cada imagen, y luego entrena un clasificador relativamente pequeño sobre esos embeddings.
- **CNN** aprende directamente desde píxeles redimensionados a un tamaño fijo. Eso la hace más autosuficiente, pero también más dependiente del tamaño del dataset, de la distribución por clase y de la calidad del entrenamiento desde cero.

En este proyecto, la CNN usa **resize directo sin padding**. La ventaja de esta decisión es la simplicidad y la reproducibilidad: todas las entradas quedan con el mismo tamaño y el pipeline es fácil de ejecutar localmente. La desventaja es que puede introducir distorsión de aspecto, lo que potencialmente perjudica clases cuyos rasgos visuales dependan más de la geometría original.

CLIP puede partir con ventaja en datasets relativamente pequeños o desbalanceados porque su encoder ya incorpora conocimiento visual aprendido en un corpus mucho mayor. En cambio, una CNN baseline necesita extraer esas regularidades únicamente a partir de las muestras del proyecto, lo que suele ser más difícil cuando hay clases muy escasas.

La escasez de varias clases también afecta la interpretación de métricas. En particular:

- **macro F1** se vuelve una señal especialmente útil porque da el mismo peso a clases frecuentes y raras,
- **weighted F1** permite ver el rendimiento global ponderado por soporte,
- y la diferencia entre ambas ayuda a entender cuánto está sufriendo el modelo en clases minoritarias.

Finalmente, las curvas de entrenamiento ayudan a interpretar estabilidad y generalización. Si un modelo mejora rápido pero luego se estanca o abre una brecha creciente entre train y val, aparecen señales de sobreajuste. Si la validación acompaña de forma más consistente, el comportamiento es más defendible desde una perspectiva académica.


In [26]:
winner_accuracy = metrics_df["test_accuracy"].idxmax()
winner_macro_f1 = metrics_df["test_macro_f1"].idxmax()
winner_weighted_f1 = metrics_df["test_weighted_f1"].idxmax()

best_acc = metrics_df.loc[winner_accuracy, "test_accuracy"]
second_acc = metrics_df.drop(index=winner_accuracy)["test_accuracy"].max()
best_macro_f1 = metrics_df.loc[winner_macro_f1, "test_macro_f1"]
second_macro_f1 = metrics_df.drop(index=winner_macro_f1)["test_macro_f1"].max()
best_weighted_f1 = metrics_df.loc[winner_weighted_f1, "test_weighted_f1"]
second_weighted_f1 = metrics_df.drop(index=winner_weighted_f1)["test_weighted_f1"].max()

classes_missing_from_val = split_summary.get("classes_missing_from_val", []) if isinstance(split_summary, dict) else []

if winner_accuracy == winner_macro_f1 == winner_weighted_f1:
    winner_sentence = f"En los artefactos analizados, **{winner_accuracy}** aparece como el mejor modelo global del Hito 2."
else:
    winner_sentence = (
        f"La comparación no arroja un ganador único en todas las métricas: accuracy favorece a **{winner_accuracy}**, "
        f"macro F1 favorece a **{winner_macro_f1}** y weighted F1 favorece a **{winner_weighted_f1}**."
    )

conclusion_md = f'''
## 11. Conclusión final del notebook

{winner_sentence}

- En **test accuracy**, la ventaja observada es de **{best_acc - second_acc:.4f}**.
- En **test macro F1**, la ventaja observada es de **{best_macro_f1 - second_macro_f1:.4f}**.
- En **test weighted F1**, la ventaja observada es de **{best_weighted_f1 - second_weighted_f1:.4f}**.

Desde una interpretación técnica, una mejor performance en **macro F1** sugiere un manejo más sólido de clases minoritarias o difíciles, mientras que una mejor **weighted F1** confirma que esa ventaja no se logra a costa del rendimiento global. Esto es especialmente importante en un dataset multiclase single-label con **35 clases** y distribución desbalanceada.

Las principales limitaciones que deben declararse en el informe son:

- existen clases con muy pocas imágenes,
- algunas clases no aparecen en validación porque no pueden repartirse en tres splits sin romper la consistencia,
- la CNN trabaja con resize fijo `128x128`, lo que simplifica el pipeline pero puede distorsionar proporciones,
- y CLIP parte desde embeddings preentrenados, por lo que la comparación enfrenta un modelo con conocimiento visual previo frente a una CNN entrenada desde píxeles en el contexto local.

En conjunto, este notebook entrega evidencia visual, tabular y metodológica suficiente para respaldar el informe y la presentación del Hito 2. Clases ausentes en validación: **{", ".join(classes_missing_from_val) if classes_missing_from_val else "ninguna"}**.
'''
display(Markdown(conclusion_md))



## 11. Conclusión final del notebook

En los artefactos analizados, **CLIP + MLP** aparece como el mejor modelo global del Hito 2.

- En **test accuracy**, la ventaja observada es de **0.1176**.
- En **test macro F1**, la ventaja observada es de **0.1645**.
- En **test weighted F1**, la ventaja observada es de **0.0982**.

Desde una interpretación técnica, una mejor performance en **macro F1** sugiere un manejo más sólido de clases minoritarias o difíciles, mientras que una mejor **weighted F1** confirma que esa ventaja no se logra a costa del rendimiento global. Esto es especialmente importante en un dataset multiclase single-label con **35 clases** y distribución desbalanceada.

Las principales limitaciones que deben declararse en el informe son:

- existen clases con muy pocas imágenes,
- algunas clases no aparecen en validación porque no pueden repartirse en tres splits sin romper la consistencia,
- la CNN trabaja con resize fijo `128x128`, lo que simplifica el pipeline pero puede distorsionar proporciones,
- y CLIP parte desde embeddings preentrenados, por lo que la comparación enfrenta un modelo con conocimiento visual previo frente a una CNN entrenada desde píxeles en el contexto local.

En conjunto, este notebook entrega evidencia visual, tabular y metodológica suficiente para respaldar el informe y la presentación del Hito 2. Clases ausentes en validación: **obj_3, obj_36, obj_37, obj_42**.
